# Exercise 1: From sentences to vectors

A warm-up for the pipeline in Exercise 2. We take a handful of sentences, turn each into a vector with a sentence-embedding model, and look at how the geometry between the vectors mirrors the meaning of the sentences.

By the end you will have seen:
- what an embedding looks like as numbers
- how cosine similarity turns two vectors into one comparable number
- which aspects of meaning the model captures well, and which it ignores

## Using this notebook

1. To run a cell, click into it and press `shift + enter`. The notebook executes the code and moves on to the next cell.
2. Cells can be run in any order and variables can be overwritten. If you lose track of the state, choose *Runtime -> Restart session* in the menu and run the cells again from the top.
3. The last expression in a cell is printed below it. To print several things, use `print()`.

Markdown cells (text) explain what the next code cell does. Details are in the code comments. Cells that say **Run the cell below** are meant to be run together with the group; **TASK** cells are for you to explore at your own pace.

## Environment setup

Run the cell below. On Colab the first line installs the one package we need; this takes about a minute.

In [ ]:
import sys
if 'google.colab' in sys.modules:  # install packages when running on Google Colab
    !pip install sentence-transformers &> /dev/null

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

## Extracting features from sentences

The cell below loads `all-MiniLM-L6-v2`, a small and fast embedding model, and extracts a vector of 384 features for each of three sentences with the `encode` method. The features are printed as a table with one row per sentence.

Run the cell below.

In [ ]:
# Define sentences
sentences = [
    "I feel great this morning",
    "I am feeling very good today",
    "I am feeling terrible",
]

# Load the pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract features
features = model.encode(sentences)

# Show the features as a table
pd.DataFrame(features, index=sentences)

**TASK 1**: Scroll through the columns. Can you see that the first two sentences have more similar numbers than either has with the third? Why would that be, given how the model was trained?

**TASK 2**: Add a fourth sentence to `sentences` that reuses one of the existing ones but swaps a word for a synonym, for instance *great* for *fantastic*. Rerun the cell. Does the new row look like its original?

## Comparing vectors with cosine similarity

Eyeballing 384 columns does not scale. Cosine similarity reduces the comparison of two vectors to one number between -1 and 1. The cell below computes it for every pair of sentences and draws the result as a heatmap.

Run the cell below.

In [ ]:
def plot_similarity(texts):
    """Embed a list of texts and draw their cosine-similarity matrix as a heatmap."""
    sims = cosine_similarity(model.encode(texts))
    plt.figure(figsize=(0.9 * len(texts) + 2, 0.8 * len(texts) + 1.5))
    sns.heatmap(sims, annot=True, fmt='.2f', vmin=0, vmax=1, cmap='viridis', square=True,
                xticklabels=texts, yticklabels=texts)
    plt.xticks(rotation=45, ha='right')
    return pd.DataFrame(sims, index=texts, columns=texts)

plot_similarity(sentences);

**TASK 3**: Add a sentence that shares words with the others but not their meaning, for example *I am feeling the table*. Does the model fall for the surface overlap?

**TASK 4**: Try `plot_similarity(["I am happy", "I am not happy", "I am sad", "I am unhappy"])`. Which pair is closest? Would a person agree? Keep this in mind for the reverse-keyed items in Exercise 2.

In [ ]:
plot_similarity(["I am happy", "I am not happy", "I am sad", "I am unhappy"]);

## Does word order matter?

The cell below compares a sentence with versions that reorder its words, including one that swaps who did what to whom.

Run the cell below.

In [ ]:
plot_similarity([
    "The dog bit the man",
    "The man bit the dog",
    "man the bit dog The",
    "A man was bitten by a dog",
    "The cat sat on the mat",
]);

**TASK 5**: Rank the four variants by their similarity to *The dog bit the man*. What does the model treat as nearly identical that you would not? One sentence: what do these embeddings capture, and what do they ignore?

**BONUS TASK**: Replace `'all-MiniLM-L6-v2'` in the model-loading cell with another model from the [sentence-transformers list](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html), for instance the multilingual `paraphrase-multilingual-MiniLM-L12-v2`, and add a sentence in another language. Does it land next to its translation?